# 04 — Temporal Features (Row-level Panel)

**Mục tiêu:** Cải thiện F1 lên **~0.75–0.80** bằng cách thêm **temporal features** trên row-level data (không aggregate — giữ 100k dòng).

**Ý tưởng cốt lõi:** Cho mỗi dòng (khách hàng C, tháng M), thêm feature từ **các tháng TRƯỚC** của cùng khách:
- Lag (giá trị tháng trước)
- Delta (thay đổi so với tháng trước)
- Rolling stats (mean, std của 3 tháng gần nhất)
- Streak (số tháng liên tiếp trả trễ)
- Trend (xu hướng tăng/giảm)

**Không leak future** vì chỉ dùng tháng nhỏ hơn. **Không mất sample** vì giữ nguyên row-level.

**So sánh mục tiêu:** notebook 02 (F1 0.682) → notebook 04 (F1 kỳ vọng ~0.75+)

## 0. Setup

In [ ]:
# Chỉ Colab
# !pip install -q lightgbm xgboost shap

In [ ]:
import json, re, warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, StandardScaler)

from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 80)

RANDOM_STATE = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Công nghệ dịch vụ tài chính')
DATA_DIR = PROJECT_DIR / 'data' / 'Credit_score_classification'
OUT_DIR = PROJECT_DIR / 'models'

# ===== Local =====
# PROJECT_DIR = Path('..')
# DATA_DIR = PROJECT_DIR / 'data' / 'Credit_score_classification'
# OUT_DIR = PROJECT_DIR / 'models'

OUT_DIR.mkdir(exist_ok=True, parents=True)

## 1. Load + Clean (reuse từ notebook 02/03)

In [ ]:
def to_num(s):
    if pd.isna(s):
        return np.nan
    cleaned = re.sub(r'[^0-9.\-]', '', str(s))
    try:
        return float(cleaned) if cleaned not in ('', '-', '.') else np.nan
    except ValueError:
        return np.nan

def parse_history(s):
    if pd.isna(s):
        return np.nan
    m = re.match(r'(\d+)\s*Years?\s*(?:and\s*(\d+)\s*Months?)?', str(s))
    if m:
        return int(m.group(1)) * 12 + (int(m.group(2)) if m.group(2) else 0)
    return np.nan

def parse_loan_types(s):
    if pd.isna(s) or s in ('', 'Not Specified'):
        return []
    parts = re.split(r',\s*(?:and\s+)?', str(s))
    return [p.strip() for p in parts if p.strip() and p.strip() != 'Not Specified']

MONTH_MAP = {'January': 1, 'February': 2, 'March': 3, 'April': 4,
             'May': 5, 'June': 6, 'July': 7, 'August': 8,
             'September': 9, 'October': 10, 'November': 11, 'December': 12}

def clean_dataset(df):
    df = df.copy()
    num_like = ['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
                'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly',
                'Monthly_Balance']
    for c in num_like:
        df[c] = df[c].apply(to_num)

    df['Credit_History_Months'] = df['Credit_History_Age'].apply(parse_history)
    df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)
    df['Payment_Behaviour'] = df['Payment_Behaviour'].replace('!@9#%8', np.nan)
    df['Occupation'] = df['Occupation'].replace('_______', np.nan)
    df['month_idx'] = df['Month'].map(MONTH_MAP)

    clip_rules = {
        'Age': (14, 100), 'Num_Bank_Accounts': (0, 20), 'Num_Credit_Card': (0, 20),
        'Interest_Rate': (0, 40), 'Num_of_Loan': (0, 20),
        'Num_of_Delayed_Payment': (0, 50), 'Num_Credit_Inquiries': (0, 30),
    }
    for col, (lo, hi) in clip_rules.items():
        df[col] = df[col].clip(lo, hi)
    return df

df = pd.read_csv(DATA_DIR / 'train.csv', low_memory=False)
df = clean_dataset(df)
print('After cleaning:', df.shape)

## 2. FE Ratios (từ notebook 03, giữ row-level)

In [ ]:
def add_ratios(df):
    df = df.copy()
    safe = lambda s: s.replace(0, np.nan)

    df['Debt_to_Income_Annual'] = df['Outstanding_Debt'] / safe(df['Annual_Income'])
    df['EMI_to_Salary_Ratio'] = df['Total_EMI_per_month'] / safe(df['Monthly_Inhand_Salary'])
    df['Investment_Rate'] = df['Amount_invested_monthly'] / safe(df['Monthly_Inhand_Salary'])
    df['CC_per_Bank'] = df['Num_Credit_Card'] / safe(df['Num_Bank_Accounts'])
    df['Loan_per_CC'] = df['Num_of_Loan'] / safe(df['Num_Credit_Card'])
    df['Age_Start_Credit'] = df['Age'] - df['Credit_History_Months'] / 12
    df['Debt_per_Loan'] = df['Outstanding_Debt'] / safe(df['Num_of_Loan'])
    df['Balance_to_Salary'] = df['Monthly_Balance'] / safe(df['Monthly_Inhand_Salary'])

    def split_beh(s):
        if pd.isna(s): return (np.nan, np.nan)
        parts = str(s).split('_')
        return (parts[0], parts[2]) if len(parts) >= 4 else (np.nan, np.nan)
    df[['Spending_Level', 'Payment_Value']] = df['Payment_Behaviour'].apply(
        lambda s: pd.Series(split_beh(s))
    )

    df['loan_list'] = df['Type_of_Loan'].apply(parse_loan_types)
    df['Num_Loan_Types'] = df['loan_list'].apply(len)
    all_types = [t for lst in df['loan_list'] for t in lst]
    top_loans = pd.Series(all_types).value_counts().head(5).index.tolist()
    for lt in top_loans:
        col_name = 'Has_' + lt.replace(' ', '_').replace('-', '_')
        df[col_name] = df['loan_list'].apply(lambda lst, t=lt: int(t in lst))
    df = df.drop(columns=['loan_list', 'Type_of_Loan'])
    return df

df = add_ratios(df)
print('After ratios:', df.shape)

## 3. Temporal Features (điểm mới của notebook này)

Sort theo `Customer_ID` + `month_idx`, sau đó với mỗi cột quan trọng tính:
- **lag_1**: giá trị tháng trước
- **delta_1**: (giá trị hiện tại) − (tháng trước)
- **roll_mean_3**: trung bình 3 tháng gần nhất (không tính tháng hiện tại)
- **roll_std_3**: độ biến động 3 tháng gần nhất

Ngoài ra: **Delay streak** (số tháng liên tiếp bị trả trễ) và **Debt trend** (slope theo tháng).

In [ ]:
TEMPORAL_COLS = ['Outstanding_Debt', 'Num_of_Delayed_Payment', 'Delay_from_due_date',
                 'Credit_Utilization_Ratio', 'Monthly_Balance', 'Changed_Credit_Limit',
                 'Debt_to_Income_Annual', 'EMI_to_Salary_Ratio']

df = df.sort_values(['Customer_ID', 'month_idx']).reset_index(drop=True)
print('Adding lag, delta, rolling features ~30s...')

gb = df.groupby('Customer_ID', sort=False)
for col in TEMPORAL_COLS:
    # Lag 1 tháng
    df[f'{col}_lag1'] = gb[col].shift(1)
    # Delta = current - lag1
    df[f'{col}_delta1'] = df[col] - df[f'{col}_lag1']
    # Rolling mean/std của 3 tháng TRƯỚC (không tính tháng hiện tại)
    df[f'{col}_roll_mean3'] = gb[col].shift(1).rolling(window=3, min_periods=1).mean().reset_index(0, drop=True)
    df[f'{col}_roll_std3'] = gb[col].shift(1).rolling(window=3, min_periods=2).std().reset_index(0, drop=True)

print('Done. Shape:', df.shape)
print('Temporal cols added:', [c for c in df.columns if any(s in c for s in ['_lag1', '_delta1', '_roll_'])])

In [ ]:
# Delay streak: số tháng LIÊN TIẾP có Delay_from_due_date > 5 (tính đến tháng trước)
print('Adding delay streak...')
df['is_late'] = (df['Delay_from_due_date'] > 5).astype(int)

def streak_up_to_prev(s):
    # s là series is_late đã sort theo month cho 1 customer
    # Với mỗi vị trí i, đếm số 1 liên tiếp kết thúc ở i-1
    shifted = s.shift(1).fillna(0).astype(int)
    grp = (shifted != shifted.shift()).cumsum()
    return shifted.groupby(grp).cumsum()

df['delay_streak_prev'] = df.groupby('Customer_ID', sort=False)['is_late'].transform(streak_up_to_prev)
df = df.drop(columns=['is_late'])
print('Done.')

In [ ]:
# Debt trend: slope của Outstanding_Debt qua các tháng đã qua
print('Adding debt trend ~30s...')

def past_slope(s):
    # s: values sorted by month. Với mỗi i, slope của s[0:i] vs range(i)
    n = len(s)
    out = np.full(n, np.nan)
    values = s.values.astype(float)
    for i in range(2, n):
        past = values[:i]
        mask = ~np.isnan(past)
        if mask.sum() >= 2:
            x = np.arange(i)[mask]
            y = past[mask]
            if x.std() > 0:
                out[i] = np.polyfit(x, y, 1)[0]
    return pd.Series(out, index=s.index)

df['debt_trend_prev'] = df.groupby('Customer_ID', sort=False)['Outstanding_Debt'].transform(past_slope)
print('Done. Final shape:', df.shape)

In [ ]:
# Xem sample của 1 customer
sample_cid = df['Customer_ID'].iloc[0]
cols_show = ['Customer_ID', 'month_idx', 'Outstanding_Debt', 'Outstanding_Debt_lag1',
             'Outstanding_Debt_delta1', 'Outstanding_Debt_roll_mean3', 'debt_trend_prev',
             'delay_streak_prev', 'Credit_Score']
df[df['Customer_ID'] == sample_cid][cols_show]

## 4. Chuẩn bị train — split theo Customer_ID

In [ ]:
drop_cols = ['ID', 'Name', 'SSN', 'Month', 'Credit_History_Age',
             'Credit_Score', 'Payment_Behaviour', 'Customer_ID']
groups = df['Customer_ID']
y_raw = df['Credit_Score']
X = df.drop(columns=drop_cols)

target_order = ['Poor', 'Standard', 'Good']
y = y_raw.map({v: i for i, v in enumerate(target_order)})

# Cột phân loại
ordinal_cols = ['Credit_Mix']
credit_mix_order = [['Bad', 'Standard', 'Good']]
onehot_cols = ['Occupation', 'Payment_of_Min_Amount', 'Spending_Level', 'Payment_Value']
onehot_cols = [c for c in onehot_cols if c in X.columns]
numeric_cols = [c for c in X.columns if c not in ordinal_cols + onehot_cols]

print(f'Total features: {X.shape[1]}')
print(f'Numeric: {len(numeric_cols)} | Ordinal: {len(ordinal_cols)} | OneHot: {len(onehot_cols)}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc', StandardScaler())
        ]), numeric_cols),
        ('ord', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('oe', OrdinalEncoder(categories=credit_mix_order,
                                   handle_unknown='use_encoded_value', unknown_value=-1))
        ]), ordinal_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), onehot_cols),
    ],
    remainder='drop'
)

# Split theo Customer_ID — tránh leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Overlap customers: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))} (phải = 0)')

## 5. Train LightGBM (cùng params notebook 02 để so sánh công bằng)

In [ ]:
def evaluate(pipe, X_te, y_te, name):
    pred = pipe.predict(X_te)
    proba = pipe.predict_proba(X_te)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='macro')
    auc = roc_auc_score(y_te, proba, multi_class='ovr', average='macro')
    print(f'{name:35s} | Acc: {acc:.4f} | F1-macro: {f1:.4f} | AUC-ovr: {auc:.4f}')
    return {'model': name, 'accuracy': acc, 'f1_macro': f1, 'auc_ovr': auc,
            'pred': pred, 'proba': proba}

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        random_state=RANDOM_STATE, verbose=-1, class_weight='balanced'
    ))
])
pipe.fit(X_train, y_train)
res = evaluate(pipe, X_test, y_test, 'LightGBM + Temporal')

## 6. Đánh giá chi tiết + so sánh notebook 02

In [ ]:
cmp = pd.DataFrame([
    {'stage': 'Notebook 02 (baseline)',   'accuracy': 0.689, 'f1_macro': 0.682, 'auc_ovr': 0.846},
    {'stage': 'Notebook 03 (aggregate)',  'accuracy': 0.590, 'f1_macro': 0.612, 'auc_ovr': 0.780},
    {'stage': 'Notebook 04 (temporal)', **{k: res[k] for k in ('accuracy', 'f1_macro', 'auc_ovr')}},
])
cmp['delta_f1_vs_nb02'] = (cmp['f1_macro'] - 0.682).round(4)
cmp

In [ ]:
print(classification_report(y_test, res['pred'], target_names=target_order))

In [ ]:
cm = confusion_matrix(y_test, res['pred'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_order, yticklabels=target_order)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — LightGBM + Temporal Features')
plt.show()

## 7. Feature Importance — kiểm chứng temporal có work không

Nếu temporal features nằm trong top 20 → chúng thực sự đóng góp.

In [ ]:
importances = pipe.named_steps['model'].feature_importances_
feat_names = pipe.named_steps['pre'].get_feature_names_out()
imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df['is_temporal'] = imp_df['feature'].apply(
    lambda s: any(k in s for k in ['_lag1', '_delta1', '_roll_', 'streak', 'trend'])
)

top = imp_df.sort_values('importance', ascending=False).head(25)
plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if t else '#3498db' for t in top['is_temporal']]
plt.barh(top['feature'][::-1], top['importance'][::-1], color=colors[::-1])
plt.title('Top 25 Feature Importance — đỏ = temporal features')
plt.tight_layout()
plt.show()

n_temporal_top = top['is_temporal'].sum()
print(f'Số temporal features trong top 25: {n_temporal_top}')

## 8. Convert Probability → FICO

In [ ]:
def proba_to_fico(proba):
    return 300 + (proba @ np.array([0, 1, 2])) / 2 * 550

def fico_to_rating(score):
    if score < 580: return 'Poor'
    if score < 670: return 'Fair'
    if score < 740: return 'Good'
    if score < 800: return 'Very Good'
    return 'Exceptional'

fico = proba_to_fico(res['proba'])
plt.figure(figsize=(10, 4))
sns.histplot(fico, bins=50, kde=True)
for x, lbl in [(580, 'Fair'), (670, 'Good'), (740, 'Very Good'), (800, 'Exceptional')]:
    plt.axvline(x, color='gray', linestyle='--', alpha=0.5)
    plt.text(x + 2, plt.ylim()[1] * 0.9, lbl, fontsize=9, color='gray')
plt.title('Phân phối FICO score — Notebook 04')
plt.show()
print(pd.Series([fico_to_rating(s) for s in fico]).value_counts())

## 9. Export — overwrite model cũ nếu tốt hơn

In [ ]:
GATE_F1 = 0.68  # chỉ overwrite nếu F1 vượt ngưỡng

if res['f1_macro'] > GATE_F1:
    joblib.dump(pipe, OUT_DIR / 'model.pkl')

    schema = {
        'task': 'classification_3class_temporal',
        'target': 'Credit_Score',
        'target_order': target_order,
        'best_model': 'LightGBM + Temporal Features',
        'metrics': {k: float(v) for k, v in {'accuracy': res['accuracy'],
                                              'f1_macro': res['f1_macro'],
                                              'auc_ovr': res['auc_ovr']}.items()},
        'input_columns': list(X.columns),
        'numeric_cols': numeric_cols,
        'ordinal_cols': ordinal_cols,
        'onehot_cols': onehot_cols,
        'credit_mix_order': credit_mix_order[0],
        'temporal_cols_base': TEMPORAL_COLS,
        'temporal_suffixes': ['_lag1', '_delta1', '_roll_mean3', '_roll_std3'],
        'extra_temporal': ['delay_streak_prev', 'debt_trend_prev'],
        'categorical_values': {
            c: sorted([str(v) for v in df[c].dropna().unique().tolist()])
            for c in onehot_cols + ordinal_cols if c in df.columns
        },
        'numeric_ranges': {
            c: {'min': float(df[c].min()), 'max': float(df[c].max()),
                'mean': float(df[c].mean()), 'median': float(df[c].median())}
            for c in numeric_cols if c in df.columns and df[c].notna().any()
        },
        'fico_conversion': {
            'formula': 'score = 300 + (0*P_poor + 1*P_std + 2*P_good) / 2 * 550',
            'bins': [
                {'max': 580, 'label': 'Poor'},
                {'max': 670, 'label': 'Fair'},
                {'max': 740, 'label': 'Good'},
                {'max': 800, 'label': 'Very Good'},
                {'max': 850, 'label': 'Exceptional'}
            ]
        },
        'note': 'Model uses temporal features (lag/delta/rolling/streak/trend). Streamlit form needs both current-month values AND past-month values or auto-compute defaults.'
    }

    with open(OUT_DIR / 'features.json', 'w', encoding='utf-8') as f:
        json.dump(schema, f, indent=2, ensure_ascii=False, default=str)

    print(f'✅ Overwrote model — F1 {res["f1_macro"]:.4f} > {GATE_F1}')
    for p in OUT_DIR.iterdir():
        print(f'  {p.name:30s} ({p.stat().st_size / 1024:.1f} KB)')
else:
    print(f'❌ Không overwrite — F1 {res["f1_macro"]:.4f} ≤ {GATE_F1}. Giữ model notebook 02.')

## 10. Sanity check

In [ ]:
loaded = joblib.load(OUT_DIR / 'model.pkl')
sample = X_test.iloc[[0]]
proba = loaded.predict_proba(sample)[0]
pred_class = target_order[loaded.predict(sample)[0]]
fico_score = proba_to_fico(proba.reshape(1, -1))[0]

print(f'Actual class:  {target_order[y_test.iloc[0]]}')
print(f'Predicted:     {pred_class}')
print(f'Probabilities: Poor={proba[0]:.3f} | Standard={proba[1]:.3f} | Good={proba[2]:.3f}')
print(f'FICO score:    {fico_score:.1f} → {fico_to_rating(fico_score)}')